In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
primaryPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
seedResults = predictions.join(evaluationRows.select('rowId', 'ndvi'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error')).group_by('model', 'featureSet', 'seed').agg(pl.col('error').abs().mean().alias('mae'), (pl.col('error').pow(2).mean().sqrt()).alias('rmse'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount')).sort('model', 'featureSet', 'seed')
seedStability = seedResults.group_by('model', 'featureSet').agg(pl.col('mae').mean().alias('meanMae'), pl.col('mae').std().alias('seedMaeStd'), pl.col('rmse').mean().alias('meanRmse'), pl.col('bias').mean().alias('meanBias'), pl.len().alias('seedCount')).sort('model', 'featureSet')
seedResults.write_parquet(primaryPath / 'seedResults.parquet')
seedStability.write_parquet(primaryPath / 'seedStability.parquet')
assert seedResults.height == 2 * 4 * 5
assert seedStability.filter(pl.col('seedCount') != 5).is_empty()
seedStability
